# GraphCast Single-Instance GNN Walkthrough

This notebook follows the **GraphCast flow from data on a longitude–latitude grid**: it builds a lat–lon grid, an icosahedral mesh, and the **Grid2Mesh** encoder graph (grid nodes → mesh nodes), then runs one `DeepTypedGraphNet` so you can inspect shapes and intermediate calls.

Use in your GraphCast env:
- `source scripts/graphcast_env.sh`
- open this notebook and run top-to-bottom


In [2]:
import inspect
from dataclasses import is_dataclass

import haiku as hk
import jax
import jax.numpy as jnp

from graphcast import deep_typed_graph_net
from graphcast import typed_graph

print('JAX backend:', jax.default_backend())
print('typed_graph module:', typed_graph.__file__)
print('DeepTypedGraphNet signature:\n ', inspect.signature(deep_typed_graph_net.DeepTypedGraphNet))
print('TypedGraph signature:\n ', inspect.signature(typed_graph.TypedGraph))
print('NodeSet signature:\n ', inspect.signature(typed_graph.NodeSet))
print('EdgeSet signature:\n ', inspect.signature(typed_graph.EdgeSet))
print('EdgesIndices signature:\n ', inspect.signature(typed_graph.EdgesIndices))


JAX backend: cpu
typed_graph module: /home/iv9432/.conda/envs/graphcast311/lib/python3.11/site-packages/graphcast/typed_graph.py
DeepTypedGraphNet signature:
  (*, node_latent_size: Mapping[str, int], edge_latent_size: Mapping[str, int], mlp_hidden_size: int, mlp_num_hidden_layers: int, num_message_passing_steps: int, num_processor_repetitions: int = 1, embed_nodes: bool = True, embed_edges: bool = True, node_output_size: Optional[Mapping[str, int]] = None, edge_output_size: Optional[Mapping[str, int]] = None, include_sent_messages_in_node_update: bool = False, use_layer_norm: bool = True, use_norm_conditioning: bool = False, activation: str = 'relu', f32_aggregation: bool = False, aggregate_edges_for_nodes_fn: str = 'segment_sum', aggregate_normalization: Optional[float] = None, name: str = 'DeepTypedGraphNet')
TypedGraph signature:
  (context: graphcast.typed_graph.Context, nodes: Mapping[str, graphcast.typed_graph.NodeSet], edges: Mapping[graphcast.typed_graph.EdgeSetKey, graphcast.

In [3]:
import numpy as np
from graphcast import icosahedral_mesh
from graphcast import grid_mesh_connectivity
from graphcast import model_utils

def tree_shapes(tree):
    def _shape(x):
        return tuple(x.shape) if hasattr(x, 'shape') else type(x).__name__
    return jax.tree_util.tree_map(_shape, tree)

# --- GraphCast flow: longitude–latitude grid -> mesh -> graph ---

# 1) Define data points on a longitude–latitude grid (e.g. coarse for toy)
resolution_deg = 10.0   # degrees between grid points
grid_lat = np.linspace(-60, 60, int(120 / resolution_deg + 1), dtype=np.float32)   # [num_lat]
grid_lon = np.linspace(0, 360, int(360 / resolution_deg), endpoint=False, dtype=np.float32)   # [num_lon]
num_grid_nodes = len(grid_lat) * len(grid_lon)
grid_nodes_lon, grid_nodes_lat = np.meshgrid(grid_lon, grid_lat)
grid_nodes_lon = grid_nodes_lon.reshape(-1)
grid_nodes_lat = grid_nodes_lat.reshape(-1)
print(f"Lat–lon grid: {len(grid_lat)} lats x {len(grid_lon)} lons = {num_grid_nodes} grid nodes")

# 2) Build icosahedral mesh hierarchy (GraphCast uses mesh for global message passing)
mesh_size = 5   # number of splits; small for toy
meshes = icosahedral_mesh.get_hierarchy_of_triangular_meshes_for_sphere(splits=mesh_size)
finest_mesh = meshes[-1]
num_mesh_nodes = finest_mesh.vertices.shape[0]
mesh_phi, mesh_theta = model_utils.cartesian_to_spherical(
    finest_mesh.vertices[:, 0], finest_mesh.vertices[:, 1], finest_mesh.vertices[:, 2])
mesh_nodes_lat, mesh_nodes_lon = model_utils.spherical_to_lat_lon(phi=mesh_phi, theta=mesh_theta)
mesh_nodes_lat = mesh_nodes_lat.astype(np.float32)
mesh_nodes_lon = mesh_nodes_lon.astype(np.float32)
print(f"Mesh: {num_mesh_nodes} nodes (finest of hierarchy with mesh_size={mesh_size})")

# 3) Grid2Mesh edges: each grid point connected to mesh nodes within a radius
def _max_edge_distance(mesh):
    senders, receivers = icosahedral_mesh.faces_to_edges(mesh.faces)
    edge_distances = np.linalg.norm(
        mesh.vertices[senders] - mesh.vertices[receivers], axis=-1)
    return float(edge_distances.max())

radius_query_fraction = 0.6
query_radius = _max_edge_distance(finest_mesh) * radius_query_fraction
grid_indices, mesh_indices = grid_mesh_connectivity.radius_query_indices(
    grid_latitude=grid_lat,
    grid_longitude=grid_lon,
    mesh=finest_mesh,
    radius=query_radius,
)
# Edges send from grid -> mesh
senders_g2m = grid_indices
receivers_g2m = mesh_indices
n_edge_g2m = senders_g2m.shape[0]
print(f"Grid2Mesh edges: {n_edge_g2m}")

# 4) Spatial features for nodes and edges (same options as GraphCast encoder)
spatial_kwargs = dict(
    add_node_positions=False,
    add_node_latitude=True,
    add_node_longitude=True,
    add_relative_positions=True,
    relative_longitude_local_coordinates=True,
    relative_latitude_local_coordinates=True,
)
senders_node_features, receivers_node_features, edge_features = model_utils.get_bipartite_graph_spatial_features(
    senders_node_lat=grid_nodes_lat,
    senders_node_lon=grid_nodes_lon,
    receivers_node_lat=mesh_nodes_lat,
    receivers_node_lon=mesh_nodes_lon,
    senders=senders_g2m,
    receivers=receivers_g2m,
    edge_normalization_factor=None,
    **spatial_kwargs,
)

# 5) Build TypedGraph for Grid2Mesh (encoder graph in full GraphCast)
n_grid_node = np.array([num_grid_nodes], dtype=np.int32)
n_mesh_node = np.array([num_mesh_nodes], dtype=np.int32)
n_edge = np.array([n_edge_g2m], dtype=np.int32)
grid_node_set = typed_graph.NodeSet(n_node=n_grid_node, features=senders_node_features)
mesh_node_set = typed_graph.NodeSet(n_node=n_mesh_node, features=receivers_node_features)
edge_set = typed_graph.EdgeSet(
    n_edge=n_edge,
    indices=typed_graph.EdgesIndices(senders=senders_g2m, receivers=receivers_g2m),
    features=edge_features,
)
nodes = {"grid_nodes": grid_node_set, "mesh_nodes": mesh_node_set}
edges = {
    typed_graph.EdgeSetKey("grid2mesh", ("grid_nodes", "mesh_nodes")): edge_set,
}
graph_in = typed_graph.TypedGraph(
    context=typed_graph.Context(n_graph=np.array([1], dtype=np.int32), features=()),
    nodes=nodes,
    edges=edges,
)

print("Input graph (Grid2Mesh from lat–lon) tree shapes:")
print(tree_shapes(graph_in))



Lat–lon grid: 13 lats x 36 lons = 468 grid nodes
Mesh: 10242 nodes (finest of hierarchy with mesh_size=5)
Grid2Mesh edges: 792
Input graph (Grid2Mesh from lat–lon) tree shapes:
TypedGraph(context=Context(n_graph=(1,), features=()), nodes={'grid_nodes': NodeSet(n_node=(1,), features=(468, 3)), 'mesh_nodes': NodeSet(n_node=(1,), features=(10242, 3))}, edges={EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes')): EdgeSet(n_edge=(1,), indices=EdgesIndices(senders=(792,), receivers=(792,)), features=(792, 4))})


In [4]:
def make_deep_typed_gnn_kwargs(overrides=None):
    overrides = overrides or {}
    sig = inspect.signature(deep_typed_graph_net.DeepTypedGraphNet)

    # Match Grid2Mesh graph from lat–lon: grid_nodes, mesh_nodes, grid2mesh edges
    defaults = {
        'latent_size': 16,
        'node_latent_size': {'grid_nodes': 16, 'mesh_nodes': 16},
        'edge_latent_size': {'grid2mesh': 16},
        'mlp_hidden_size': 32,
        'mlp_num_hidden_layers': 1,
        'num_message_passing_steps': 3,
        'num_processor_repetitions': 2,
        'use_layer_norm': True,
        'name': 'grid2mesh_gnn',
    }
    defaults.update(overrides)

    kwargs = {}
    missing_required = []
    for pname, p in sig.parameters.items():
        if pname == 'self':
            continue
        if pname in defaults:
            kwargs[pname] = defaults[pname]
        elif p.default is inspect._empty:
            missing_required.append(pname)

    if missing_required:
        raise ValueError(
            'Please provide overrides for required args not covered by defaults: '
            + ', '.join(missing_required)
        )
    return kwargs

gnn_kwargs = make_deep_typed_gnn_kwargs()
print('DeepTypedGraphNet kwargs used:')
print(gnn_kwargs)


DeepTypedGraphNet kwargs used:
{'node_latent_size': {'grid_nodes': 16, 'mesh_nodes': 16}, 'edge_latent_size': {'grid2mesh': 16}, 'mlp_hidden_size': 32, 'mlp_num_hidden_layers': 1, 'num_message_passing_steps': 3, 'num_processor_repetitions': 2, 'use_layer_norm': True, 'name': 'grid2mesh_gnn'}


**When and how model weights are initialized**

- **When:** Weights are created and initialized when `transformed.init(rng, graph_in)` is called below. Haiku runs the forward pass once in "init" mode and allocates parameters for every `hk.Linear`, `hk.LayerNorm`, etc., inside `DeepTypedGraphNet`.

- **How:** `DeepTypedGraphNet` does not pass custom initializers. It uses `hk.nets.MLP` (and `hk.LayerNorm`) with Haiku’s defaults:
  - **Linear layers:** Variance scaling (Glorot/Xavier): weights drawn so that variance is scale/fan_in (scale=1.0 by default).
  - **LayerNorm:** scale and offset are initialized to 1 and 0 (when `create_scale`/`create_offset` are True).
- The **RNG** used for sampling initial weights is the one passed to `init(rng, graph_in)` (e.g. `jax.random.PRNGKey(42)`). Same RNG ⇒ same initial weights; change the key to get a different random init.

In [5]:
trace_records = []

def capture_interceptor(next_f, args, kwargs, context):
    out = next_f(*args, **kwargs)
    module_name = context.module.module_name
    method_name = context.method_name

    # Keep a lightweight trace of every Haiku method call shape.
    trace_records.append({
        'module': module_name,
        'method': method_name,
        'output_shapes': tree_shapes(out),
    })
    return out

def forward(graph):
    model = deep_typed_graph_net.DeepTypedGraphNet(**gnn_kwargs)
    return model(graph)

transformed = hk.transform(forward)
rng = jax.random.PRNGKey(42)

with hk.intercept_methods(capture_interceptor):
    params = transformed.init(rng, graph_in)
    graph_out = transformed.apply(params, rng, graph_in)

print('Output graph tree shapes:')
print(tree_shapes(graph_out))
print('Number of captured module calls:', len(trace_records))


Output graph tree shapes:
TypedGraph(context=Context(n_graph=(1,), features=()), nodes={'grid_nodes': NodeSet(n_node=(1,), features=(468, 16)), 'mesh_nodes': NodeSet(n_node=(1,), features=(10242, 16))}, edges={EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes')): EdgeSet(n_edge=(1,), indices=EdgesIndices(senders=(792,), receivers=(792,)), features=(792, 16))})
Number of captured module calls: 354


In [6]:
# --- Initial vs final data (input graph → output graph) ---

def _summary(arr, name, max_rows=2):
    """Print shape and a small sample of array values."""
    a = arr if hasattr(arr, 'shape') else None
    if a is None:
        print(f"  {name}: (no array)")
        return
    print(f"  {name}: shape={a.shape}", end="")
    flat = jnp.reshape(a, (-1,))
    n = min(5, flat.size)
    if n > 0:
        sample = jnp.asarray(flat[:n])
        print(f"  sample[:{n}]={sample}")
    else:
        print()

print("=== INITIAL (input) graph ===")
print(tree_shapes(graph_in))
for node_name, node_set in graph_in.nodes.items():
    _summary(node_set.features, f"nodes['{node_name}'].features")
for edge_key, edge_set in graph_in.edges.items():
    _summary(edge_set.features, f"edges[{edge_key!r}].features")

print("\n=== FINAL (output) graph ===")
print(tree_shapes(graph_out))
for node_name, node_set in graph_out.nodes.items():
    _summary(node_set.features, f"nodes['{node_name}'].features")
for edge_key, edge_set in graph_out.edges.items():
    _summary(edge_set.features, f"edges[{edge_key!r}].features")

=== INITIAL (input) graph ===
TypedGraph(context=Context(n_graph=(1,), features=()), nodes={'grid_nodes': NodeSet(n_node=(1,), features=(468, 3)), 'mesh_nodes': NodeSet(n_node=(1,), features=(10242, 3))}, edges={EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes')): EdgeSet(n_edge=(1,), indices=EdgesIndices(senders=(792,), receivers=(792,)), features=(792, 4))})
  nodes['grid_nodes'].features: shape=(468, 3)  sample[:5]=[-0.8660254  1.         0.        -0.8660254  0.9848077]
  nodes['mesh_nodes'].features: shape=(10242, 3)  sample[:5]=[ 0.18759258  0.49999997  0.86602545  0.7946544  -0.50000006]
  edges[EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes'))].features: shape=(792, 4)  sample[:5]=[ 0.91277814 -0.01030706  0.82069737 -0.39939138  0.91277796]

=== FINAL (output) graph ===
TypedGraph(context=Context(n_graph=(1,), features=()), nodes={'grid_nodes': NodeSet(n_node=(1,), features=(468, 16)), 'mesh_nodes': NodeSet(n_node=(1,), features=(10242, 16))

In [18]:
print(graph_in[1]['grid_nodes'].features)
print(graph_out[1]['grid_nodes'].features)

[[-0.8660254   1.          0.        ]
 [-0.8660254   0.9848077   0.17364818]
 [-0.8660254   0.9396926   0.34202012]
 ...
 [ 0.8660254   0.8660253  -0.5000002 ]
 [ 0.8660254   0.93969256 -0.34202036]
 [ 0.8660254   0.9848077  -0.17364843]]
[[-1.5205731  -3.4971485  -0.19070393 ...  3.4219303   1.9127841
  -0.2975538 ]
 [-1.5698546  -3.655401   -0.17419457 ...  3.4288962   2.0260158
  -0.30699664]
 [-1.5897081  -3.7932098  -0.15744573 ...  3.4568233   2.1025593
  -0.3433138 ]
 ...
 [-3.6308968  -2.8848083  -5.86606    ...  3.1588326   4.432462
   0.8258986 ]
 [-3.390684   -2.6976953  -5.7615557  ...  3.4160862   4.012253
   0.7370787 ]
 [-3.3132014  -2.5073526  -5.649602   ...  3.636654    3.793834
   0.6279096 ]]


In [7]:
# Split trace: __init__ runs during init (output None); rest are from apply (real shapes)
init_records = [r for r in trace_records if r['method'] == '__init__']
apply_records = [r for r in trace_records if r['method'] != '__init__']
print(f'Total: {len(trace_records)} calls ({len(init_records)} init, {len(apply_records)} apply/forward)\n')

# Show first N apply-time calls (these have real tensor shapes)
N = 40
print('First N apply-time calls (forward pass — real shapes):')
for i, rec in enumerate(apply_records[:N]):
    print(f"[{i:03d}] {rec['module']}.{rec['method']}")
    print('      output:', rec['output_shapes'])

# GNN-related calls: match grid2mesh_gnn and submodules (not "typed"/"graph")
gnn_calls = [r for r in apply_records if 'grid2mesh_gnn' in r['module'] or 'gnn' in r['module'].lower()]
print('\nFiltered GNN-ish (apply) calls:', len(gnn_calls))
for i, rec in enumerate(gnn_calls[:N]):
    print(f"[{i:03d}] {rec['module']}.{rec['method']}")
    print('      output:', rec['output_shapes'])


Total: 354 calls (122 init, 232 apply/forward)

First N apply-time calls (forward pass — real shapes):
[000] grid2mesh_gnn._networks_builder
      output: ('function', ['function', 'function', 'function'], 'function')
[001] grid2mesh_gnn/~_networks_builder/encoder_edges_grid2mesh_mlp/~/linear_0.__call__
      output: (792, 32)
[002] grid2mesh_gnn/~_networks_builder/encoder_edges_grid2mesh_mlp/~/linear_1.__call__
      output: (792, 16)
[003] grid2mesh_gnn/~_networks_builder/encoder_edges_grid2mesh_mlp.__call__
      output: (792, 16)
[004] grid2mesh_gnn/~_networks_builder/encoder_edges_grid2mesh_layer_norm.__call__
      output: (792, 16)
[005] grid2mesh_gnn/~_networks_builder/sequential.__call__
      output: (792, 16)
[006] grid2mesh_gnn/~_networks_builder/encoder_nodes_grid_nodes_mlp/~/linear_0.__call__
      output: (468, 32)
[007] grid2mesh_gnn/~_networks_builder/encoder_nodes_grid_nodes_mlp/~/linear_1.__call__
      output: (468, 16)
[008] grid2mesh_gnn/~_networks_builder/encoder

## Notes

- If a constructor fails, inspect the printed signatures and adjust fallback kwargs in `make_tiny_typed_graph` / `make_deep_typed_gnn_kwargs`.
- Increase `num_message_passing_steps` to see deeper propagation.
- You can replace random node/edge features with a real single weather sample once your data-to-typed-graph mapping is ready.
